# Model Training

This notebook trains and evaluates multiple machine learning models for fraud detection using the engineered dataset. 
The workflow includes data preparation, experiment tracking with MLflow, model training, performance evaluation, and model selection. The best-performing model will be used in the explainability stage of the proposed Explainable AI (XAI) fraud detection framework.
# Logistic Regression
# Random Forest
# XGBoost
# Model Evaluation

In [1]:
# Data manipulation
import pandas as pd
import numpy as np

# Data preprocessing
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Machine learning models
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

# Model evaluation
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report
)

# Experiment tracking
import mlflow
import mlflow.sklearn

# Model persistence
import joblib

# Timing
import time

# Visualisation
import matplotlib.pyplot as plt
import seaborn as sns

# Display settings
pd.set_option("display.max_columns", None)

# Loading Datase

In [3]:
# Load the engineered dataset
data = pd.read_csv("../Data/engineered_fraud_dataset.csv")

# Display the first five rows
data.head()

,transaction_amount,login_attempts,device_risk_score,transfer_frequency,anomaly_score,account_age_days,transaction_time_hour,failed_transactions_last_30d,avg_monthly_balance,daily_transaction_count,geo_distance_km,session_duration_minutes,transaction_velocity_score,payment_channel,authentication_type,card_present_flag,international_transaction_flag,suspicious_ip_flag,fraud_flag,new_account,high_value_transaction,high_login_attempts,high_velocity,night_transaction,far_distance_transaction,high_device_risk
0,17829.01,4,12.0,13,0.37,2354,22,25,112760.07,63,3189,92,71.8,POS Terminal,OTP,1,1,1,0,0,0,0,0,1,0,0
1,16401.83,1,34.3,17,0.26,3181,17,15,118899.52,83,839,63,11.8,Web Banking,Biometric,0,0,1,0,0,0,0,0,0,0,0
2,9678.29,8,67.8,39,0.15,1390,3,2,408168.98,9,3938,80,35.7,ATM,OTP,1,0,1,0,0,0,0,0,1,0,0
3,19013.38,5,17.8,42,0.55,3716,16,6,80771.69,78,11111,11,74.8,Mobile App,OTP,1,1,0,0,0,1,0,0,0,0,0
4,13834.95,3,88.9,63,0.24,4694,16,10,382265.32,17,3171,87,0.0,Mobile App,OTP,1,0,0,0,0,0,0,0,0,0,1


## Dataset Overview

The engineered dataset is loaded and inspected to confirm that all features have been successfully imported and are ready for model training. This step verifies the dataset structure, identifies any missing values, and confirms the data types of each variable.

In [4]:
# Display the first five rows
data.head()

# Display dataset dimensions
print("Dataset Shape:", data.shape)

# Display column names
print("Columns:")
print(data.columns.tolist())

# Check for missing values
print("Missing Values:")
print(data.isnull().sum())

# Check data types
print("Data Types:")
print(data.dtypes)

Dataset Shape: (10000, 26)
Columns:
['transaction_amount', 'login_attempts', 'device_risk_score', 'transfer_frequency', 'anomaly_score', 'account_age_days', 'transaction_time_hour', 'failed_transactions_last_30d', 'avg_monthly_balance', 'daily_transaction_count', 'geo_distance_km', 'session_duration_minutes', 'transaction_velocity_score', 'payment_channel', 'authentication_type', 'card_present_flag', 'international_transaction_flag', 'suspicious_ip_flag', 'fraud_flag', 'new_account', 'high_value_transaction', 'high_login_attempts', 'high_velocity', 'night_transaction', 'far_distance_transaction', 'high_device_risk']
Missing Values:
transaction_amount                0
login_attempts                    0
device_risk_score                 0
transfer_frequency                0
anomaly_score                     0
account_age_days                  0
transaction_time_hour             0
failed_transactions_last_30d      0
avg_monthly_balance               0
daily_transaction_count           0


## Identify Feature Types

The dataset contains both numerical and categorical variables. Identifying feature types is necessary before applying preprocessing techniques such as one-hot encoding and feature scaling.

In [5]:
# Separate numerical and categorical columns
numerical_cols = data.select_dtypes(include=['int64', 'float64']).columns.tolist()
categorical_cols = data.select_dtypes(include=['object']).columns.tolist()

print("Numerical Features:")
print(numerical_cols)

print("\nCategorical Features:")
print(categorical_cols)

Numerical Features:
['transaction_amount', 'login_attempts', 'device_risk_score', 'transfer_frequency', 'anomaly_score', 'account_age_days', 'transaction_time_hour', 'failed_transactions_last_30d', 'avg_monthly_balance', 'daily_transaction_count', 'geo_distance_km', 'session_duration_minutes', 'transaction_velocity_score', 'card_present_flag', 'international_transaction_flag', 'suspicious_ip_flag', 'fraud_flag', 'new_account', 'high_value_transaction', 'high_login_attempts', 'high_velocity', 'night_transaction', 'far_distance_transaction', 'high_device_risk']

Categorical Features:
['payment_channel', 'authentication_type']


## Encode Categorical Variables

Machine learning algorithms require numerical input. Therefore, the categorical variables are transformed into numerical representations using one-hot encoding.

In [6]:
# One-hot encode categorical variables
data = pd.get_dummies(
    data,
    columns=categorical_cols,
    drop_first=True
)

print("Dataset Shape After Encoding:", data.shape)

data.head()

Dataset Shape After Encoding: (10000, 30)


,transaction_amount,login_attempts,device_risk_score,transfer_frequency,anomaly_score,account_age_days,transaction_time_hour,failed_transactions_last_30d,avg_monthly_balance,daily_transaction_count,geo_distance_km,session_duration_minutes,transaction_velocity_score,card_present_flag,international_transaction_flag,suspicious_ip_flag,fraud_flag,new_account,high_value_transaction,high_login_attempts,high_velocity,night_transaction,far_distance_transaction,high_device_risk,payment_channel_Mobile App,payment_channel_POS Terminal,payment_channel_Web Banking,authentication_type_OTP,authentication_type_Password Only,authentication_type_Two-Factor Authentication
0,17829.01,4,12.0,13,0.37,2354,22,25,112760.07,63,3189,92,71.8,1,1,1,0,0,0,0,0,1,0,0,False,True,False,True,False,False
1,16401.83,1,34.3,17,0.26,3181,17,15,118899.52,83,839,63,11.8,0,0,1,0,0,0,0,0,0,0,0,False,False,True,False,False,False
2,9678.29,8,67.8,39,0.15,1390,3,2,408168.98,9,3938,80,35.7,1,0,1,0,0,0,0,0,1,0,0,False,False,False,True,False,False
3,19013.38,5,17.8,42,0.55,3716,16,6,80771.69,78,11111,11,74.8,1,1,0,0,0,1,0,0,0,0,0,True,False,False,True,False,False
4,13834.95,3,88.9,63,0.24,4694,16,10,382265.32,17,3171,87,0.0,1,0,0,0,0,0,0,0,0,0,1,True,False,False,True,False,False


In [7]:
bool_columns = data.select_dtypes(include='bool').columns

data[bool_columns] = data[bool_columns].astype(int)

print(data.head())

   transaction_amount  login_attempts  device_risk_score  transfer_frequency  \
0            17829.01               4               12.0                  13   
1            16401.83               1               34.3                  17   
2             9678.29               8               67.8                  39   
3            19013.38               5               17.8                  42   
4            13834.95               3               88.9                  63   

   anomaly_score  account_age_days  transaction_time_hour  \
0           0.37              2354                     22   
1           0.26              3181                     17   
2           0.15              1390                      3   
3           0.55              3716                     16   
4           0.24              4694                     16   

   failed_transactions_last_30d  avg_monthly_balance  daily_transaction_count  \
0                            25            112760.07                   

## Define Features and Target Variable

The dataset is divided into input features (X) and the target variable (y). The target variable is `fraud_flag`, while all remaining variables are used as predictors for fraud detection.

In [8]:
# Define features and target variable
X = data.drop('fraud_flag', axis=1)
y = data['fraud_flag']

print("Feature Matrix Shape:", X.shape)
print("Target Variable Shape:", y.shape)

Feature Matrix Shape: (10000, 29)
Target Variable Shape: (10000,)


In [ ]:
#checking the distribution of the target variable

print("Target Variable Distribution:")
print(y.value_counts())

print("\nTarget Variable Percentage:")
print(y.value_counts(normalize=True) * 100)

Target Variable Distribution:
fraud_flag
0    8749
1    1251
Name: count, dtype: int64

Target Variable Percentage:
fraud_flag
0    87.49
1    12.51
Name: proportion, dtype: float64


## Train-Test Split

The dataset is divided into training and testing subsets using an 80:20 ratio. Stratified sampling is applied to preserve the class distribution of fraudulent and non-fraudulent transactions in both datasets.

In [10]:
from sklearn.model_selection import train_test_split

# Split the data
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

# Display dataset sizes
print("Training Features:", X_train.shape)
print("Testing Features:", X_test.shape)
print("Training Target:", y_train.shape)
print("Testing Target:", y_test.shape)

Training Features: (8000, 29)
Testing Features: (2000, 29)
Training Target: (8000,)
Testing Target: (2000,)
